# Day 11 — Spark Structured Streaming Fundamentals

**Structured Streaming **is Spark's stream-processing engine. 
It allows us to process **incoming data incrementally **using familiar DataFrame and SQL operations.
Instead of loading an entire dataset each time, Spark can **process new data as it becomes available.**

### Batch V/s Streaming
````
**Batch processing	                Structured Streaming**
Processes a bounded dataset	        Processes a continuously growing dataset
Uses spark.read	                    Uses spark.readStream
Uses df.write	                    Uses df.writeStream
Executes a finite job	            Can run continuously or until available data is processed
````

Structured Streaming typically uses** micro-batches**, where Spark processes newly available data in small batches.
 when new data is available --> it create micro batches and processes the file in incremental way


Spark does not automatically launch another concurrent micro-batch for that same query after 10 seconds. The next batch starts when the current batch finishes, potentially causing processing lag.

### Trigger
Triggers control when Spark attempts to process available data.
Trigger	Meaning
availableNow=True	Processes available data and stops
processingTime="10 seconds"	Attempts a micro-batch approximately every 10 seconds
Default trigger	Processes data as quickly as possible using the engine's default scheduling behavior

.trigger(availableNow=True)

# Delta-to-Delta streaming pipeline

In [0]:
%sql
select * from  workspace.customer360_bronze.transactions_source;

select * from workspace.customer360_bronze.transactions_stream;

## Step 1- Create a source Delta table

In [0]:
%python

# Create DataFrame with sample data
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField('transactionId', IntegerType(), False),
    StructField('custId', IntegerType(), False),
    StructField('serviceType', StringType(), False),
    StructField('amount', IntegerType(), False)
])

transaction_data = (
    (1, 1001, "Wireless", 120.50),
    (2, 1002, "Fiber", 145.99),
    (3, 1001, "Internet", 75.00),
    (4, 1003, "Wireless", 85.25)
)

dfTransaction = spark.createDataFrame(transaction_data, schema)
dfTransaction.write.format("delta").mode("overwrite").saveAsTable("workspace.customer360_bronze.transactions_source")

## Step 2- Read the Delta table as a stream

In [0]:
dfStream = (
    spark.readStream
    .format("delta")
    .table(
        "workspace.customer360_bronze.transactions_source"
    )
)


## Step 3 — Write to a target Delta table

In [0]:
%python

checkpoint_path = (
    "/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/transactions_day11"
)

# Ingest into Bronze
query = (
    dfStream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(
        "workspace.customer360_bronze.transactions_stream"
    )
)

query.awaitTermination()

## Step 4- verify the result

In [0]:
%sql
SELECT *
FROM workspace.customer360_bronze.transactions_stream
ORDER BY transactionId;

## Step 5- Add new transactions

execute step 5 and then step 2 and step 3
so you will be adding 2 records to source table and write them to stream table. 

In [0]:
%python

new_transactionData = [
        (5, 1004, "Fiber", 110.00),
        (6, 1001, "Wireless", 95.00)
]

dfNewTransactions = spark.createDataFrame(new_transactionData, schema)
dfNewTransactions.write.format("delta").mode("append").saveAsTable("workspace.customer360_bronze.transactions_source")

## Mini Challenge

In [0]:
from pyspark.sql import functions as F

dfCitySummary = (
    spark.table(
        "workspace.customer360_bronze.transactions_stream"
    )
    .groupBy("custId")
    .agg(
        F.count("*").alias("transactionCount"),
        F.sum("amount").alias("totalAmount")
    ).
)

display(dfCitySummary)

### What is Spark Structured Streaming, and how does it work?
Spark Structured Streaming is a stream-processing engine that allows us to process incoming data incrementally using DataFrame and SQL APIs. It typically processes data in micro-batches, where each batch handles newly available records. We use checkpoints to track processing progress and support recovery if the job fails. Depending on the use case, we can run the stream continuously or use the AvailableNow trigger for scheduled incremental processing.

````
**Step	Action	                            Result**
1	    Create source Delta table	        4 transactions 
2	    Read source using readStream	    Streaming DataFrame created 
3	    Write using writeStream	Initial     4 records ingested
4	    Query target	                    Initial records verified
5	    Append 2 transactions to source	    Source now has 6 records
6	    Rerun Steps 2 and 3	                Target now has 6 records
````

# Part 2: Streaming Aggregations & Output Modes

## 1. Batch aggregation vs streaming aggregation

In [0]:
from pyspark.sql import functions as F

dfSummary = (
    spark.table(
        "workspace.customer360_bronze.transactions_stream"
    )
    .groupBy("custId")
    .agg(
        F.count("*").alias("transactionCount"),
        F.sum("amount").alias("totalAmount")
    ).orderBy("custId")
)

display(dfSummary)

## 2. Understanding output modes
This is one of the most frequently misunderstood Structured Streaming concepts.

**Append** --> Only newly finalized output rows
Suitable for simple streaming ingestion. For aggregations, it generally requires a watermark and a supported event-time window so Spark knows when a result is final.

**Update** --> Only rows updated since the previous trigger
For customer 1001, Spark can output a revised running total when a new transaction arrives.

**Complete** --> Entire aggregation result after each trigger

Spark outputs the current results for every customer, including customers whose totals haven't changed.

**_Visual example_**

Suppose our current aggregation is:
```
custId	totalAmount
1001	195
1002	145
1003	85
```

A new transaction of $95 arrives for customer 1001.
```
**Update mode**
custId	totalAmount
1001	290

Only changed customer
````

````
**Complete mode**
custId	totalAmount
1001	290
1002	145
1003	85

Entire updated result
````



**Important**: Output mode describes which rows a streaming query emits. It doesn't automatically specify how a destination Delta table should update existing records.

For example, if you want to maintain a customer-summary Delta table with one row per customer, you may need foreachBatch and MERGE rather than simply appending updated aggregates.



## 3. Hands-on — Streaming customer aggregations

### Step 1 — Create a streaming aggregation

In [0]:
from pyspark.sql import functions as F

source_table = ("workspace.customer360_bronze.transactions_stream")

dfStream = (
    spark.readStream
    .format("delta")
    .table(source_table)
)

dfCustomerAgg = (
    dfStream
    .groupBy("custId")
    .agg(
        F.count("*").alias("transactionCount"),
        F.sum("amount").alias("totalAmount")
    )
)

maintains the aggregation state for each customer.

With outputMode("complete"), Spark emits the entire current aggregation result after each micro-batch.

That's why you see all four customers in the output.

### Step 2 — Run the aggregation in Complete mode

******The memory sink is intended for learning and debugging. It isn't a durable production destination.**

In [0]:
checkpoint_path = (
    "/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/customer_summary_day11"
)

query = (
    dfCustomerAgg.writeStream
    .format("memory")
    .queryName("customer_summary_day11")
    .outputMode("complete")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()


### Step 3 — View the results

In [0]:
%sql
SELECT *
FROM customer_summary_day11
ORDER BY custId;

### Step 4 — Test Append mode

In [0]:
queryAppend = (
    dfCustomerAgg.writeStream
    .format("memory")
    .queryName("customer_summary_append_day11")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/customer360_bronze/"
        "ingestion_metadata/checkpoints/customer_summary_append_day11"
    )
    .trigger(availableNow=True)
    .start()
)

queryAppend.awaitTermination()

**Expected**: An error similar to:-Append output mode not supported for streaming aggregations without watermark
Spark should reject this query because you're performing an aggregation without a watermark or other mechanism to finalize its output.

This is intentional. We haven't configured an event-time watermark or window that would allow Spark to finalize the aggregation.

**Why?** --> Because a customer's total can continue changing whenever another transaction arrives. Spark cannot treat** the aggregate as final.**

### Step 5 — Update mode
Use another independent checkpoint:

In [0]:
queryUpdate = (
    dfCustomerAgg.writeStream
    .format("memory")
    .queryName("customer_summary_update_day11")
    .outputMode("update")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/customer_summary_update_day11"
    )
    .trigger(availableNow=True)
    .start()
)

queryUpdate.awaitTermination()

In [0]:
%sql
SELECT *
FROM customer_summary_update_day11
ORDER BY custId;

### **4. Why do we need watermarks?**
Imagine a transaction happened at 10:00 AM, but it arrived in your pipeline at 10:20 AM.

This is called **late-arriving data.**

## Next exercise — Update vs Complete mode

### Step 1 — Add one new transaction
Append this record to your original source table:

In [0]:
# Create DataFrame with sample data
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

new_transaction = [
    (7, 1001, "Wireless", 50.00)
]

schema = StructType([
    StructField('transactionId', IntegerType(), False),
    StructField('custId', IntegerType(), False),
    StructField('serviceType', StringType(), False),
    StructField('amount', IntegerType(), False)
])

dfNew = spark.createDataFrame(new_transaction, schema)

dfNew.write.format("delta").mode("append").saveAsTable("workspace.customer360_bronze.transactions_source")

In [0]:
%sql
select * from workspace.customer360_bronze.transactions_source;

Now run cell 7, 9, 11 to add data to transaction_stream table. 

Conceptually, after the new transaction arrives:
````
Update mode
custId	totalAmount
1001	340

Only the changed customer group is emitted.
````

````
Complete mode
custId	totalAmount
1001	340
1002	145
1003	85
1004	110

All current customer groups are emitted.
````

This is the key certification concept: Update emits changed aggregation results; Complete emits the entire aggregation result.

## Next: Day 11 — Part 3: Watermarks and Late-Arriving Data

# Day 11 — Part 3: Event-Time Windows, Watermarks & Late-Arriving Data


## Create event-time transactions

### Step 1 — Create the source table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, DoubleType, StringType
)

event_schema = StructType([
    StructField("transactionId", IntegerType(), False),
    StructField("custId", IntegerType(), False),
    StructField("amount", DoubleType(), False),
    StructField("eventTime", StringType(), False)
])

data = [
    (1, 1001, 100.0, "2026-11-12 10:01:00"),
    (2, 1001, 50.0,  "2026-11-12 10:03:00"),
    (3, 1001, 75.0,  "2026-11-12 10:07:00"),
    (4, 1002, 200.0, "2026-11-12 10:02:00")
]

df = (
    spark.createDataFrame(data, event_schema)
    .withColumn("eventTime", F.to_timestamp("eventTime"))
)

df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.customer360_bronze.transactions_event_source"
)

### Step 2 — Read the source as a stream

In [0]:
dfEventStream = (
    spark.readStream
    .format("delta")
    .table(
        "workspace.customer360_bronze.transactions_event_source"
    )
)

### Step 3 — Apply a watermark and window aggregation

Here:
- **withWatermark("eventTime", "10 minutes")** configures the event-time lateness threshold.
- **window("eventTime", "5 minutes")** creates non-overlapping 5-minute windows.
- **groupBy** groups by both window and customer.
- **count** and **sum** maintain the aggregation state.

In [0]:
dfWindowAgg = (
    dfEventStream
    .withWatermark("eventTime", "10 minutes")
    .groupBy(
        F.window("eventTime", "5 minutes"),
        F.col("custId")
    )
    .agg(
        F.count("*").alias("transactionCount"),
        F.sum("amount").alias("totalAmount")
    )
)

### Step 4 — Execute in Update mode

We'll use foreachBatch to inspect the rows emitted by each micro-batch.
**foreachBatch** --> only to print the micro-batch output. It doesn't create a persistent aggregation table.

In [0]:
def show_microbatch(batch_df, batch_id): 
    print(f"Micro-batch: {batch_id}")

    batch_df.select(
        "custId",
        F.col("window.start").alias("windowStart"),
        F.col("window.end").alias("windowEnd"),
        "transactionCount",
        "totalAmount"
    ).orderBy("custId", "windowStart").show(
        truncate=False
    )

checkpoint_path = ("/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/transactions_window_update_day11")

query = (
    dfWindowAgg.writeStream
    .outputMode("update")
    .foreachBatch(show_microbatch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
print("Query status:", query.status)
print("Last progress:", query.lastProgress)
print("Exception:", query.exception())

In [0]:
for progress in query.recentProgress:
    print(
        "Batch:", progress.get("batchId"),
        "Input rows:", progress.get("numInputRows"),
        "Watermark:", progress.get("eventTime", {}).get("watermark")
    )

### Step 4- ---> USE ME and discard the above,

### Better approach — Save micro-batch output to a Delta table
Instead of relying on print() or .show(), let's write the aggregation results to a separate Delta table.

In [0]:
from pyspark.sql import functions as F

history_table = (
    "workspace.customer360_bronze."
    "transactions_window_updates_day11"
)

history_checkpoint = (
    "/Volumes/workspace/customer360_bronze/"
    "ingestion_metadata/checkpoints/"
    "transactions_window_history_day11"
)

def save_microbatch(batch_df, batch_id):
    (
        batch_df
        .select(
            "custId",
            F.col("window.start").alias("windowStart"),
            F.col("window.end").alias("windowEnd"),
            "transactionCount",
            "totalAmount"
        )
        .withColumn("batchId", F.lit(batch_id))
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(history_table)
    )

queryHistory = (
    dfWindowAgg.writeStream
    .outputMode("update")
    .foreachBatch(save_microbatch)
    .option("checkpointLocation", history_checkpoint)
    .trigger(availableNow=True)
    .start()
)

queryHistory.awaitTermination()

In [0]:
%sql
SELECT *
FROM workspace.customer360_bronze.transactions_window_updates_day11
ORDER BY batchId, windowStart, custId;

### 3. Introduce a late-arriving transaction

### Step 6 — Append a late event

In [0]:
late_data = [
    (5, 1001, 25.0, "2026-11-12 10:04:00")
]

dfLate = (
    spark.createDataFrame(late_data, event_schema)
    .withColumn("eventTime", F.to_timestamp("eventTime"))
)

dfLate.write.format("delta").mode("append").saveAsTable(
    "workspace.customer360_bronze.transactions_event_source"
)

### 4. Simulate an event that's too late

### Step 7 — Append a future event and then run Step4

In [0]:
future_data = [
    (6, 1002, 300.0, "2026-11-12 10:30:00")
]

dfFuture = (
    spark.createDataFrame(future_data, event_schema)
    .withColumn("eventTime", F.to_timestamp("eventTime"))
)

dfFuture.write.format("delta").mode("append").saveAsTable(
    "workspace.customer360_bronze.transactions_event_source"
)

### Step 8 — Append another old transaction
After confirming that the watermark has advanced, append:

In [0]:
too_late_data = [
    (7, 1001, 40.0, "2026-11-12 10:02:00")
]

dfTooLate = (
    spark.createDataFrame(too_late_data, event_schema)
    .withColumn("eventTime", F.to_timestamp("eventTime"))
)

dfTooLate.write.format("delta").mode("append").saveAsTable(
    "workspace.customer360_bronze.transactions_event_source"
)

### How does Spark Structured Streaming handle late-arriving data?
Spark Structured Streaming supports event-time processing using watermarks. We can define a watermark threshold, such as 10 minutes, to control how long Spark maintains state for supported event-time aggregations. If a transaction arrives late but its window is still open, Spark can incorporate it into the aggregation. Once the watermark advances beyond the window, Spark can finalize that window and discard subsequent late records.